### Random Sample Generation:
The function generate_subset_sum_case generates an instance of the multiset, while the function save_uniform_test_cases utilizes the former to generate multiple random test cases of a requested set size and number of cases to be generated, it writes those cases in a unified format to a txt file.

This random sample generator will be mainly used for performance testing of both the Brute Force and Heuristic algorithms.

In [1]:
import random
from pathlib import Path 

def generate_subset_sum_case(T, n):
    """
    Generates a multiset of n random integers where each element <= T.
    """
    lower_bound = 1 if T >= 1 else T
    multiset = [random.randint(lower_bound, T) for _ in range(n)]
    return multiset

def save_uniform_test_cases(filename, numCases, caseSize, min_T, max_T):
    """
    Generates numCases test cases, each precisely of size caseSize.
    The target sum T is randomized for each case between min_T and max_T.
    """
    with open(filename, 'w') as f:
        # Write the total number of test cases at the top
        f.write(f"{numCases}\n")
        
        for _ in range(numCases):
            # Generate a random target sum T for this specific case
            T = random.randint(min_T, max_T)
            
            # Generate the multiset using the fixed caseSize
            multiset = generate_subset_sum_case(T, caseSize)
            
            # Unified Format:
            # Line 1: T
            # Line 2: space-separated elements
            f.write(f"{T}\n")
            f.write(" ".join(map(str, multiset)) + "\n")



## generating the uniform tests for brute force and heuristic

In [ ]:
# --- Execution Script ---

# we are going to generate 20 instances of each set size n to compute the mean time performance.
# we are going to generate 20 instances of the probelm


num_cases_to_generate = 30


# Bounds for the random target sum T
minimum_target = 20
maximum_target = 300

for i in range(0, 21): # generating 21 files, corresponding to 21 sizes
    output_filename = "uniform_test_size" + str(i)+ ".txt" # create corresponding name
    target_dir = Path("../tests")
    file_path = target_dir / output_filename
    save_uniform_test_cases(
        filename= file_path,
        numCases= num_cases_to_generate,
        caseSize= i,
        min_T=minimum_target,
        max_T=maximum_target
    )





print(f"Generated {num_cases_to_generate} test cases (of sizes 0 to {i}).")

### Brute Force Algorithm Implementation:

The following function finds all subsets within a set of a specific size, it does not account for the set being a multiSet, so there might be duplicate sets outputed, but that will not affect the correctness.

In [2]:
def findAllSubSets(mySet, size): # find all size-subsets of mySet, assuming that the size is valid
  #if size > len(mySet) or size < 0: # if size is invalid for a subset
  #  return -1 # return -1
  totalSubsets = [] # declare an empty list of
  if(size == 0): # if we want 0-size subsets
    return [totalSubsets] # there is only the empty set
  elif (size == 1): # all 1-sized subsets
    for elm in mySet: # for all elements in the set
      totalSubsets.append([elm]) # append singleton sets

  elif (size == len(mySet)): # if we want subsets of the exact size of the set
    totalSubsets = [mySet]

  else: # if the size requested is non of the above
    for ind in range(len(mySet)): # for each index in mySet

      exclusionSet = findAllSubSets(mySet[ind+1:], size-1) # compute the exclusion set
      for Set in exclusionSet: # append the excluded element to the set
        Set.append(mySet[ind])

      totalSubsets = [*totalSubsets, *exclusionSet] # append that set to the totalsets
  return totalSubsets




The following helper function aggregates the powerset used to find the possible set whose elements add up to the desired sum, the space complexity involves the call to the function for all valid sizes, the outputed set stores 2^n sets, the largest of which contains n elements, the largest set contain n elements --> O(n*2^n) space

In [7]:
def generatePowerSet(mySet):
    n = len(mySet) # compute the size of your multiset
    subsets = [] # initialize an empty subset
    for size in range((2**(n)+1)): # for all valid sizes of a subset
        subsets = [*subsets, *findAllSubSets(mySet, size)]
    return subsets

The following code implements the brute force algorithm for finding the subset that adds up to the sum

In [8]:
def ExhaustiveAlgorithm(desiredSum, multiset):
  powerset = generatePowerSet(multiset)
  for subset in powerset:
    sum = 0
    for elm in subset:
      sum += elm # is this part critical to space complexity? we have one sum variable that we store in the data and that we constantly discard
    if(sum == desiredSum):
      print(subset)
      return subset
  return ["NOT FOUND"]


### Running Brute Force Algorithm:

The following script extracts the test cases recorded in the txt file and runs the algorithm to perform the initial testing specified in part 5.1

In [ ]:
import collections
from pathlib import Path

def verify_result(desiredSum, original_multiset, returned_subset):
    """
    Validates the correctness of the returned subset against constraints.
    Throws an Exception if any validation check fails.
    """
    # If the algorithm determined no subset exists, no subset validation is needed
    if returned_subset == ["NOT FOUND"]:
        return

    # Check 1: Verify the mathematical sum matches the target
    if sum(returned_subset) != desiredSum:
        raise ValueError(
            f"Validation Failed: Subset sum {sum(returned_subset)} "
            f"does not match desired sum {desiredSum}."
        )

    # Check 2: Verify elements and duplicate counts using frequency mapping
    # Count frequencies in the original multiset
    original_counts = {}
    for item in original_multiset:
        original_counts[item] = original_counts.get(item, 0) + 1

    # Count frequencies in the returned subset
    subset_counts = {}
    for item in returned_subset:
        subset_counts[item] = subset_counts.get(item, 0) + 1

    # Validate each element in the returned subset against the original counts
    for item, count in subset_counts.items():
        if item not in original_counts:
            raise ValueError(
                f"Validation Failed: Element {item} in the returned subset "
                f"does not exist in the original multiset."
            )
        if count > original_counts[item]:
            raise ValueError(
                f"Validation Failed: Element {item} appears {count} times "
                f"in the subset, but only {original_counts[item]} times in the original multiset."
            )


def run_initial_testing(input_filename, output_filename):
    """
    Reads test cases from input_filename, runs ExhaustiveAlgorithm,
    validates correctness, and logs results to output_filename.
    Robustly handles files of size 0 where data rows are omitted.
    """
    result_dir = Path("../results")
    file_path = result_dir / output_filename
    
    # Ensure the directory exists before writing to it
    result_dir.mkdir(parents=True, exist_ok=True)

    with open(input_filename, 'r') as infile:
        lines = [line.strip() for line in infile if line.strip()]
    
    if not lines:
        print(f"Input file '{input_filename}' is empty.")
        return

    # First line contains the total number of test cases
    num_cases = int(lines[0])
    
    results_to_write = []
    line_index = 1

    # CRITICAL FIX: Detect if this is a uniform test file of size 0.
    # If it is size 0, the file only contains target sums (1 line per case + 1 header line)
    is_size_zero_file = (len(lines) == num_cases + 1)

    for case_num in range(1, num_cases + 1):
        # Ensure we don't read past the file limits
        if line_index >= len(lines):
            break

        # Parse Line 1 of the case: Target Sum (T)
        T = int(lines[line_index])
        line_index += 1
        
        # Parse Line 2: Elements (Only if the file actually contains array data)
        if is_size_zero_file:
            multiset = []
        else:
            if line_index < len(lines):
                multiset = list(map(int, lines[line_index].split()))
                line_index += 1
            else:
                multiset = []
        
        print(f"Running Test Case {case_num}: T = {T}, Size = {len(multiset)}")

        # Execute the algorithm
        result_subset = ExhaustiveAlgorithm(T, multiset)

        # Run verification checks (Will raise exception and pause execution if invalid)
        verify_result(T, multiset, result_subset)

        # Determine the status flag (YES if found, NO if "NOT FOUND")
        status = "YES" if result_subset != ["NOT FOUND"] else "NO"
        
        # Save structural details to write out later
        results_to_write.append((T, multiset, status, result_subset))

    # Write structural results to the output file
    with open(file_path, 'w') as outfile:
        # Top line matches total count
        outfile.write(f"{len(results_to_write)}\n")
        
        for T, multiset, status, result_subset in results_to_write:
            # Format: 
            # Line 1: T STATUS
            # Line 2: original multiset elements
            # Line 3: elements of the found subset (or NOT FOUND)
            outfile.write(f"{T} {status}\n")
            outfile.write(" ".join(map(str, multiset)) + "\n")
            outfile.write(" ".join(map(str, result_subset)) + "\n")


# --- Main Execution Loop ---
import re


try:
    dir = Path("../tests") 
    if dir.exists() and dir.is_dir():
        
        # 1. Grab all files into a list
        file_list = [path for path in dir.iterdir() if path.is_file()]
        
        # 2. Sort them numerically by pulling out the numbers inside the filename
        # This finds any sequence of digits (\d+) in the name and converts it to an int for sorting
        file_list.sort(key=lambda p: [int(x) for x in re.findall(r'\d+', p.name)])
        
        # 3. Loop over the beautifully sorted list
        for path in file_list:
            results_file = "result_" + path.name
            run_initial_testing(path, results_file)
            print(f"All tests completed and verified successfully for {path.name}.")
            print(f"Results logged to '{results_file}'.\n")
    else:
        print("Error: '../tests' directory not found.")
    
except ValueError as e:
    print(f"\n[EXECUTION PAUSED] Algorithmic Error Found during verification:")
    print(e)

### create initial general tests (section 3)

Generate 15 random tests with varying sizes (1-18) for quick smoke testing.

In [3]:
# Generate 15 random tests with varying sizes (1-18) in one file
output_filename = "../tests/initial_general_tests.txt"

with open(output_filename, 'w') as f:
    # Write total number of tests
    f.write("15\n")
    
    # Generate 15 tests with random sizes from 1 to 18
    for i in range(15):
        # Random size between 1 and 18
        test_size = random.randint(1, 18)
        
        # Random target sum between 20 and 300
        target_sum = random.randint(20, 300)
        
        # Generate multiset using the existing function
        multiset = generate_subset_sum_case(target_sum, test_size)
        
        # Write in format: size, target, elements
        f.write(f"{test_size}\n")
        f.write(f"{target_sum}\n")
        f.write(" ".join(map(str, multiset)) + "\n")

print("Generated 15 mixed-size tests in initial_general_tests.txt")

Generated 15 mixed-size tests in initial_general_tests.txt


### Heuristic Algorithm Implementation:

In [ ]:
def rgli_subset_sum(numbers, target, max_trials=40, seed=None):
    """
    Randomized Greedy with Local Improvement (RGLI)
    for the optimization version of Subset Sum.

    Returns:
        best_subset, best_sum
    """
    if seed is not None:
        random.seed(seed)

    best_subset = []
    best_sum = 0

    for _ in range(max_trials):
        # Phase 1: randomized greedy
        shuffled = numbers[:]
        random.shuffle(shuffled)

        current_subset = []
        current_sum = 0

        for a in shuffled:
            if current_sum + a <= target:
                current_subset.append(a)
                current_sum += a

        # Phase 2: local improvement
        improved = True

        while improved:
            improved = False

            selected = current_subset[:]
            random.shuffle(selected)
            unselected = numbers[:]

            # remove selected items from unselected
            for x in selected:
                unselected.remove(x)

            for chosen in selected:
                remaining_capacity = target - (current_sum - chosen)

                # find largest unselected item that fits after removing chosen
                candidates = [x for x in unselected if x <= remaining_capacity]

                if candidates:
                    replacement = max(candidates)

                    if replacement > chosen:
                        current_subset.remove(chosen)
                        current_subset.append(replacement)

                        current_sum = current_sum - chosen + replacement

                        improved = True
                        break

        # update global best
        if current_sum > best_sum:
            best_sum = current_sum
            best_subset = current_subset[:]

        if best_sum == target:
            break

    return best_subset, best_sum

### run initial tests on heuristic algorithm